# Belief Rating Prediction – MovieLens Beliefs Dataset

**3 Research Questions:**
- **RQ1** – Belief Gap by Genre: Do users over/underestimate movies compared to community ratings, and how does this vary by genre?
- **RQ2** – Belief Formation Decomposition: What drives belief ratings – personal tendency (user bias), movie reputation (item popularity), or complex user-item interactions (latent factors)?
- **RQ3** – Belief as Predictor: Does belief rating add value for predicting actual ratings?

---
## Section 0 – Setup & Load Data

Install dependencies, configure DATA_DIR, load CSV files.

If data files are not found, the notebook prints a guidance message and skips computation cells — it will **not** crash.

In [ ]:
# ── Install dependencies (Colab) ─────────────────────────────────────────
# Uncomment the line below if running on Google Colab:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────
import os
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### 0.1 – Configure DATA_DIR

Edit `DATA_DIR` to point to the folder containing CSV files.

**Google Drive:** mount Drive and set e.g. `DATA_DIR = "/content/drive/MyDrive/data"`.  
**Local upload:** use Colab file upload and set `DATA_DIR = "/content"`.

In [ ]:
# ── [CONFIGURE ME] ────────────────────────────────────────────────────────
DATA_DIR = "data"   # ← change to your data folder path

### 0.2 – Load CSV files

Expected files in `DATA_DIR`:
- `belief_data.csv`
- `user_rating_history.csv`
- `movies.csv`
- `user_recommendation_history.csv` (optional)
- `movie_elicitation_set.csv` (optional)

In [ ]:
def load_csv(filename, **kwargs):
    """Load a CSV from DATA_DIR; return None with a guidance message if missing."""
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"[INFO] {filename} not found at {path}. "
              f"Place the file in DATA_DIR='{DATA_DIR}' and re-run.")
        return None
    df = pd.read_csv(path, **kwargs)
    print(f"  Loaded {filename}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df

print(f"Loading data from: {DATA_DIR}")
belief_raw      = load_csv("belief_data.csv")
ratings_raw     = load_csv("user_rating_history.csv")
movies_raw      = load_csv("movies.csv")
rec_history     = load_csv("user_recommendation_history.csv")
elicitation_set = load_csv("movie_elicitation_set.csv")

DATA_AVAILABLE = all(df is not None for df in [belief_raw, ratings_raw, movies_raw])
if DATA_AVAILABLE:
    print("\n✓ Core data loaded successfully.")
else:
    print("\n⚠ Some core files missing. Computation cells will be skipped.")

In [ ]:
if DATA_AVAILABLE:
    print("=== belief_data sample ===")
    display(belief_raw.head(3))
    print("\n=== user_rating_history sample ===")
    display(ratings_raw.head(3))
    print("\n=== movies sample ===")
    display(movies_raw.head(3))

---
## Section 1 – EDA & Data Visualization

- Distribution of `isSeen`, `userPredictRating`, `userCertainty`
- Top genres bar chart
- Rating distribution from `user_rating_history`
- Correlation heatmap
- Trend by `month_idx` (if elicitation set available)

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping EDA section.")
else:
    print("belief_data columns:", belief_raw.columns.tolist())
    print("user_rating_history columns:", ratings_raw.columns.tolist())
    print("movies columns:", movies_raw.columns.tolist())
    print("\nisSeen value counts:")
    print(belief_raw["isSeen"].value_counts())

In [ ]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1) isSeen distribution
    ax = axes[0]
    belief_raw["isSeen"].value_counts().sort_index().plot(kind="bar", ax=ax, color=sns.color_palette("muted"))
    ax.set_title("Distribution of isSeen", fontsize=13)
    ax.set_xlabel("isSeen"); ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

    # 2) userPredictRating distribution (isSeen=0 only)
    ax = axes[1]
    seen0 = belief_raw[belief_raw["isSeen"] == 0]["userPredictRating"].dropna()
    ax.hist(seen0, bins=18, color=sns.color_palette("muted")[1], edgecolor="white")
    ax.set_title("userPredictRating (isSeen=0)", fontsize=13)
    ax.set_xlabel("Predicted Rating"); ax.set_ylabel("Count")

    # 3) userCertainty distribution
    ax = axes[2]
    if "userCertainty" in belief_raw.columns:
        belief_raw["userCertainty"].value_counts().sort_index().plot(kind="bar", ax=ax,
            color=sns.color_palette("muted")[2])
        ax.set_title("Distribution of userCertainty", fontsize=13)
        ax.set_xlabel("Certainty Level"); ax.set_ylabel("Count")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    else:
        ax.text(0.5, 0.5, "userCertainty not available", ha="center", va="center")

    plt.tight_layout()
    plt.show()

In [ ]:
if DATA_AVAILABLE:
    # Top genres from movies_raw
    genres_expanded = (
        movies_raw["genres"]
        .str.split("|")
        .explode()
        .str.strip()
    )
    genres_expanded = genres_expanded[genres_expanded != "(no genres listed)"]
    top_genres = genres_expanded.value_counts().head(15)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_genres.values, y=top_genres.index, palette="muted")
    plt.title("Top 15 Movie Genres", fontsize=14)
    plt.xlabel("Count"); plt.ylabel("Genre")
    plt.tight_layout()
    plt.show()

In [ ]:
if DATA_AVAILABLE and elicitation_set is not None and "month_idx" in elicitation_set.columns:
    monthly = elicitation_set.groupby("month_idx").size().reset_index(name="count")
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=monthly, x="month_idx", y="count", marker="o")
    plt.title("Number of Elicitations per Month", fontsize=13)
    plt.xlabel("Month Index"); plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
if DATA_AVAILABLE:
    belief_seen0 = belief_raw[belief_raw["isSeen"] == 0].copy()
    num_cols = belief_seen0.select_dtypes(include=np.number).columns.tolist()
    if len(num_cols) >= 2:
        corr = belief_seen0[num_cols].corr()
        plt.figure(figsize=(10, 7))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                    linewidths=0.5, square=True)
        plt.title("Correlation Heatmap – belief_data (isSeen=0)", fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
if DATA_AVAILABLE:
    plt.figure(figsize=(8, 4))
    ratings_raw["rating"].value_counts().sort_index().plot(kind="bar",
        color=sns.color_palette("muted")[3], edgecolor="white")
    plt.title("Rating Distribution – user_rating_history", fontsize=13)
    plt.xlabel("Rating"); plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

---
## Section 2 – Preprocessing & Feature Engineering

1. Filter belief rows (`isSeen=0`, `userPredictRating` not null)
2. One-hot encode genres from `movies.csv`
3. Compute `user_stats` (mean, count, std) from rating history
4. Compute `movie_stats` (mean, count) from rating history
5. Merge all features
6. Time-based per-user split (70/15/15)

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping preprocessing section.")
else:
    # ── Step 1: Filter belief rows ─────────────────────────────────────────
    belief = belief_raw[
        (belief_raw["isSeen"] == 0) &
        (belief_raw["userPredictRating"].notna())
    ].copy()
    print(f"Belief rows (isSeen=0, userPredictRating not null): {len(belief):,}")

    # ── Step 2: Merge movie genres ─────────────────────────────────────────
    movies_genres = movies_raw[["movieId", "genres"]].copy()
    all_genres = (
        movies_genres["genres"].str.split("|").explode().str.strip()
        .pipe(lambda s: s[s != "(no genres listed)"])
        .unique().tolist()
    )
    GENRE_COLS = sorted(all_genres)

    for g in GENRE_COLS:
        movies_genres[f"genre_{g}"] = movies_genres["genres"].str.contains(g, regex=False).astype(int)

    belief = belief.merge(
        movies_genres.drop(columns=["genres"]),
        on="movieId", how="left"
    )

    # ── Step 3: User stats from rating history ─────────────────────────────
    user_stats = (
        ratings_raw.groupby("userId")["rating"]
        .agg(user_mean_rating="mean", user_rating_count="count", user_rating_std="std")
        .reset_index()
    )
    belief = belief.merge(user_stats, on="userId", how="left")

    # ── Step 4: Movie stats from rating history ────────────────────────────
    movie_stats = (
        ratings_raw.groupby("movieId")["rating"]
        .agg(movie_mean_rating="mean", movie_rating_count="count")
        .reset_index()
    )
    belief = belief.merge(movie_stats, on="movieId", how="left")

    print(f"Merged belief dataframe: {belief.shape}")
    print(f"Genre columns: {len(GENRE_COLS)}")

In [ ]:
if DATA_AVAILABLE:
    # ── Step 5: Time-based per-user split ──────────────────────────────────
    def time_based_split(df, tstamp_col="tstamp", train_ratio=0.70, val_ratio=0.15):
        """
        Per-user chronological split → train / val / test DataFrames.
        Users with fewer than 3 rows are assigned entirely to train.
        """
        train_rows, val_rows, test_rows = [], [], []
        for user_id, group in df.groupby("userId"):
            group_sorted = group.sort_values(tstamp_col)
            n = len(group_sorted)
            if n < 3:
                train_rows.append(group_sorted)
                continue
            n_train = max(1, int(n * train_ratio))
            n_val   = max(1, int(n * val_ratio))
            train_rows.append(group_sorted.iloc[:n_train])
            val_rows.append(group_sorted.iloc[n_train:n_train + n_val])
            test_rows.append(group_sorted.iloc[n_train + n_val:])

        train_df = pd.concat(train_rows, ignore_index=True)
        val_df   = pd.concat(val_rows,   ignore_index=True) if val_rows   else pd.DataFrame()
        test_df  = pd.concat(test_rows,  ignore_index=True) if test_rows  else pd.DataFrame()
        return train_df, val_df, test_df

    tstamp_col = "tstamp" if "tstamp" in belief.columns else belief.columns[0]
    train_df, val_df, test_df = time_based_split(belief, tstamp_col=tstamp_col)
    print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
# ── Evaluation helpers ────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

CLIP_MIN, CLIP_MAX = 0.5, 5.0

def clip_preds(preds):
    """Clip predictions to valid rating range [0.5, 5.0]."""
    return np.clip(preds, CLIP_MIN, CLIP_MAX)

def evaluate(name, y_true, y_pred):
    y_pred_clipped = clip_preds(y_pred)
    return {
        "model": name,
        "RMSE": round(rmse(y_true, y_pred_clipped), 4),
        "MAE":  round(mae(y_true, y_pred_clipped), 4),
    }

---
## Section 3 – RQ1: Belief Gap Analysis

> *Do users tend to expect higher or lower ratings than actual community ratings, and how does this differ by genre?*

**Belief gap** = `userPredictRating` − `movie_mean_actual_rating`
- Positive gap → user overestimates (expects more than community gives)
- Negative gap → user underestimates (expects less than community gives)

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping RQ1.")
else:
    # Merge actual movie mean rating into belief
    rq1_df = belief.copy()
    if "movie_mean_rating" not in rq1_df.columns:
        rq1_df = rq1_df.merge(movie_stats[["movieId", "movie_mean_rating"]], on="movieId", how="left")

    rq1_df["belief_gap"] = rq1_df["userPredictRating"] - rq1_df["movie_mean_rating"]
    rq1_df_valid = rq1_df.dropna(subset=["belief_gap"])

    print(f"RQ1: {len(rq1_df_valid):,} belief rows with known movie actual rating")
    print(f"\nOverall belief gap statistics:")
    print(rq1_df_valid["belief_gap"].describe().round(4))
    gap_mean = rq1_df_valid["belief_gap"].mean()
    print(f"\nMean belief gap: {gap_mean:+.3f} → users ", end="")
    print("OVERESTIMATE" if gap_mean > 0 else "UNDERESTIMATE", "on average")

In [ ]:
if DATA_AVAILABLE and len(rq1_df_valid) > 0:
    # Compute mean belief_gap per genre
    genre_gaps = {}
    for g in GENRE_COLS:
        col = f"genre_{g}"
        if col in rq1_df_valid.columns:
            subset = rq1_df_valid[rq1_df_valid[col] == 1]["belief_gap"]
            if len(subset) >= 5:
                genre_gaps[g] = subset.mean()

    genre_gap_df = pd.DataFrame.from_dict(genre_gaps, orient="index", columns=["mean_belief_gap"])
    genre_gap_df = genre_gap_df.sort_values("mean_belief_gap")

    colors = ["#d73027" if v < 0 else "#1a9850" for v in genre_gap_df["mean_belief_gap"]]

    plt.figure(figsize=(12, 6))
    bars = plt.barh(genre_gap_df.index, genre_gap_df["mean_belief_gap"], color=colors)
    plt.axvline(0, color="black", linewidth=1.2, linestyle="--")
    plt.xlabel("Mean Belief Gap (userPredictRating − actual mean)", fontsize=12)
    plt.title("RQ1: Belief Gap by Genre\n(Red = underestimate, Green = overestimate)", fontsize=14)
    plt.tight_layout()
    plt.show()

    print("\nTop 5 most overestimated genres:")
    print(genre_gap_df.tail(5)[::-1].round(3).to_string())
    print("\nTop 5 most underestimated genres:")
    print(genre_gap_df.head(5).round(3).to_string())

In [ ]:
if DATA_AVAILABLE and "userCertainty" in rq1_df_valid.columns:
    cert_gap = (
        rq1_df_valid.groupby("userCertainty")["belief_gap"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    cert_gap.columns = ["userCertainty", "mean_gap", "std_gap", "count"]

    plt.figure(figsize=(8, 4))
    colors_cert = ["#d73027" if v < 0 else "#1a9850" for v in cert_gap["mean_gap"]]
    plt.bar(cert_gap["userCertainty"].astype(str), cert_gap["mean_gap"], color=colors_cert)
    plt.axhline(0, color="black", linewidth=1.0, linestyle="--")
    plt.xlabel("Certainty Level"); plt.ylabel("Mean Belief Gap")
    plt.title("RQ1: Belief Gap by Certainty Level", fontsize=13)
    plt.tight_layout()
    plt.show()
    print(cert_gap.to_string(index=False))

---
## Section 4 – RQ2: Belief Formation Decomposition

> *What drives belief ratings – personal tendency (user bias), movie reputation (item popularity), or complex user-item interactions (latent factors)?*

We answer this by **incrementally adding components** and measuring RMSE reduction at each level:

| Level | Model | Component added |
|-------|-------|-----------------|
| 0 | Global Mean | Baseline – no personalization |
| 1 | + User Bias | Personal tendency |
| 2 | + Item Bias | Movie reputation |
| 3 | BiasedMF (SGD) | Latent user-item interactions |
| 4 | SVD++ | Implicit feedback from rating history |

The RMSE waterfall chart shows which component contributes most to explaining beliefs.

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping RQ2.")
else:
    TARGET = "userPredictRating"
    y_train = train_df[TARGET].values
    y_val   = val_df[TARGET].values
    y_test  = test_df[TARGET].values
    rq2_results = []

In [ ]:
if DATA_AVAILABLE:
    # ── Level 0: Global Mean ───────────────────────────────────────────────
    global_mean = y_train.mean()
    pred_gm_val = np.full(len(y_val), global_mean)
    pred_gm_tst = np.full(len(y_test), global_mean)

    res = evaluate("L0: Global Mean (val)",  y_val,  pred_gm_val)
    rq2_results.append({**res, "split": "val",  "level": 0})
    res = evaluate("L0: Global Mean (test)", y_test, pred_gm_tst)
    rq2_results.append({**res, "split": "test", "level": 0})
    print(f"Level 0 – Global Mean: Val RMSE={rq2_results[0]['RMSE']:.4f}  Test RMSE={rq2_results[1]['RMSE']:.4f}")

In [ ]:
if DATA_AVAILABLE:
    # ── Level 1: + User Bias ───────────────────────────────────────────────
    user_mean_map = train_df.groupby("userId")[TARGET].mean().to_dict()

    def predict_user_mean(df):
        return np.array([user_mean_map.get(uid, global_mean) for uid in df["userId"]])

    res = evaluate("L1: User Bias (val)",  y_val,  predict_user_mean(val_df))
    rq2_results.append({**res, "split": "val",  "level": 1})
    res = evaluate("L1: User Bias (test)", y_test, predict_user_mean(test_df))
    rq2_results.append({**res, "split": "test", "level": 1})
    print(f"Level 1 – User Bias: Val RMSE={rq2_results[-2]['RMSE']:.4f}  Test RMSE={rq2_results[-1]['RMSE']:.4f}")

In [ ]:
if DATA_AVAILABLE:
    # ── Level 2: + Item Bias (Movie Mean) & Additive Bias ─────────────────
    movie_mean_map = train_df.groupby("movieId")[TARGET].mean().to_dict()

    def predict_movie_mean(df):
        return np.array([movie_mean_map.get(mid, global_mean) for mid in df["movieId"]])

    res = evaluate("L2a: Movie Mean (val)",  y_val,  predict_movie_mean(val_df))
    rq2_results.append({**res, "split": "val",  "level": 2})
    res = evaluate("L2a: Movie Mean (test)", y_test, predict_movie_mean(test_df))
    rq2_results.append({**res, "split": "test", "level": 2})

    # Additive bias: r̂ = μ + b_u + b_i  (VECTORIZED – no iterrows)
    mu   = global_mean
    b_u  = {uid: (m - mu) for uid, m in user_mean_map.items()}
    b_i  = {mid: (m - mu) for mid, m in movie_mean_map.items()}

    def predict_additive_bias(df):
        """Vectorized additive bias prediction."""
        bu_vals = df["userId"].map(b_u).fillna(0).values
        bi_vals = df["movieId"].map(b_i).fillna(0).values
        return np.clip(mu + bu_vals + bi_vals, CLIP_MIN, CLIP_MAX)

    res = evaluate("L2b: Additive Bias (val)",  y_val,  predict_additive_bias(val_df))
    rq2_results.append({**res, "split": "val",  "level": 2})
    res = evaluate("L2b: Additive Bias (test)", y_test, predict_additive_bias(test_df))
    rq2_results.append({**res, "split": "test", "level": 2})
    print(f"Level 2 – Additive Bias: Val RMSE={rq2_results[-2]['RMSE']:.4f}  Test RMSE={rq2_results[-1]['RMSE']:.4f}")

In [ ]:
class BiasedMF:
    """
    Biased Matrix Factorization trained with SGD.

    Formula: r̂_ui = μ + b_u + b_i + p_u^T q_i

    Parameters
    ----------
    n_factors : int   – dimension of latent factors
    n_epochs  : int   – max training epochs
    lr        : float – learning rate (eta)
    reg       : float – L2 regularization (lambda)
    patience  : int   – early stopping patience (epochs without val improvement)
    clip_min, clip_max : float – prediction clipping range
    random_state : int
    """

    def __init__(self, n_factors=20, n_epochs=25, lr=0.005, reg=0.02,
                 patience=5, clip_min=0.5, clip_max=5.0, random_state=42):
        self.n_factors = n_factors
        self.n_epochs  = n_epochs
        self.lr        = lr
        self.reg       = reg
        self.patience  = patience
        self.clip_min  = clip_min
        self.clip_max  = clip_max
        self.rng       = np.random.default_rng(random_state)

    def _init_params(self, n_users, n_items):
        k = self.n_factors
        self.bu_ = np.zeros(n_users)
        self.bi_ = np.zeros(n_items)
        self.P_  = self.rng.normal(0, 0.1, (n_users, k))
        self.Q_  = self.rng.normal(0, 0.1, (n_items, k))

    def fit(self, train_df, val_df=None, target_col="userPredictRating",
            user_col="userId", item_col="movieId"):
        """Fit BiasedMF with SGD + early stopping on val RMSE."""
        all_users = pd.concat([
            train_df[user_col],
            val_df[user_col] if val_df is not None else train_df[user_col]
        ]).unique()
        all_items = pd.concat([
            train_df[item_col],
            val_df[item_col] if val_df is not None else train_df[item_col]
        ]).unique()

        self.user2idx_ = {u: i for i, u in enumerate(all_users)}
        self.item2idx_ = {m: i for i, m in enumerate(all_items)}

        self.mu_ = train_df[target_col].mean()
        self._init_params(len(all_users), len(all_items))

        u_arr = train_df[user_col].map(self.user2idx_).values
        i_arr = train_df[item_col].map(self.item2idx_).values
        r_arr = train_df[target_col].values.astype(float)

        self.train_rmse_history_ = []
        self.val_rmse_history_   = []

        best_val_rmse = float("inf")
        best_bu, best_bi, best_P, best_Q = None, None, None, None
        no_improve = 0

        for epoch in tqdm(range(self.n_epochs), desc="Training BiasedMF"):
            idx = self.rng.permutation(len(r_arr))
            u_s, i_s, r_s = u_arr[idx], i_arr[idx], r_arr[idx]

            for u_idx, i_idx, r in zip(u_s, i_s, r_s):
                pred = self.mu_ + self.bu_[u_idx] + self.bi_[i_idx] + self.P_[u_idx] @ self.Q_[i_idx]
                err  = r - pred
                self.bu_[u_idx] += self.lr * (err - self.reg * self.bu_[u_idx])
                self.bi_[i_idx] += self.lr * (err - self.reg * self.bi_[i_idx])
                pu_old = self.P_[u_idx].copy()
                self.P_[u_idx] += self.lr * (err * self.Q_[i_idx] - self.reg * self.P_[u_idx])
                self.Q_[i_idx] += self.lr * (err * pu_old          - self.reg * self.Q_[i_idx])

            train_preds = self._predict_batch(train_df, user_col, item_col)
            self.train_rmse_history_.append(rmse(r_arr, train_preds))

            if val_df is not None and len(val_df) > 0:
                y_val_ep  = val_df[target_col].values
                val_preds = self._predict_batch(val_df, user_col, item_col)
                val_r = rmse(y_val_ep, clip_preds(val_preds))
                self.val_rmse_history_.append(val_r)

                if val_r < best_val_rmse - 1e-5:
                    best_val_rmse = val_r
                    best_bu = self.bu_.copy()
                    best_bi = self.bi_.copy()
                    best_P  = self.P_.copy()
                    best_Q  = self.Q_.copy()
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= self.patience:
                        print(f"  Early stopping at epoch {epoch+1} (best val RMSE={best_val_rmse:.4f})")
                        break

        # Restore best parameters
        if best_bu is not None:
            self.bu_, self.bi_, self.P_, self.Q_ = best_bu, best_bi, best_P, best_Q

        return self

    def _predict_batch(self, df, user_col="userId", item_col="movieId"):
        """Vectorized batch prediction (no Python for-loop)."""
        u_idxs = df[user_col].map(self.user2idx_).fillna(-1).astype(int).values
        i_idxs = df[item_col].map(self.item2idx_).fillna(-1).astype(int).values
        preds = np.full(len(df), self.mu_)

        valid = (u_idxs >= 0) & (i_idxs >= 0)
        u_v, i_v = u_idxs[valid], i_idxs[valid]
        preds[valid] = (self.mu_ + self.bu_[u_v] + self.bi_[i_v]
                        + np.sum(self.P_[u_v] * self.Q_[i_v], axis=1))

        only_u = (u_idxs >= 0) & (i_idxs < 0)
        preds[only_u] = self.mu_ + self.bu_[u_idxs[only_u]]

        only_i = (u_idxs < 0) & (i_idxs >= 0)
        preds[only_i] = self.mu_ + self.bi_[i_idxs[only_i]]

        return np.clip(preds, self.clip_min, self.clip_max)

    def predict(self, df, user_col="userId", item_col="movieId"):
        """Return clipped predictions."""
        return self._predict_batch(df, user_col, item_col)

In [ ]:
class SVDpp:
    """
    SVD++ (Koren, 2008) – Biased MF + implicit feedback.

    Formula: r̂_ui = μ + b_u + b_i + (p_u + |N(u)|^{-½} Σ_{j∈N(u)} y_j)^T q_i

    N(u) = set of items user has rated (implicit signal from rating history).

    Parameters
    ----------
    n_factors : int
    n_epochs  : int
    lr        : float
    reg       : float
    patience  : int  – early stopping patience
    clip_min, clip_max : float
    random_state : int
    """

    def __init__(self, n_factors=20, n_epochs=25, lr=0.005, reg=0.02,
                 patience=5, clip_min=0.5, clip_max=5.0, random_state=42):
        self.n_factors = n_factors
        self.n_epochs  = n_epochs
        self.lr        = lr
        self.reg       = reg
        self.patience  = patience
        self.clip_min  = clip_min
        self.clip_max  = clip_max
        self.rng       = np.random.default_rng(random_state)

    def fit(self, train_df, val_df=None,
            target_col="userPredictRating",
            user_col="userId", item_col="movieId",
            implicit_df=None):
        """
        Fit SVD++ with SGD + early stopping.

        implicit_df : DataFrame with columns [user_col, item_col] representing
                      items each user has interacted with (rating history).
                      If None, uses train_df interactions as implicit signal.
        """
        all_users = pd.concat([
            train_df[user_col],
            val_df[user_col] if val_df is not None else train_df[user_col]
        ]).unique()
        all_items = pd.concat([
            train_df[item_col],
            val_df[item_col] if val_df is not None else train_df[item_col]
        ]).unique()

        self.user2idx_ = {u: i for i, u in enumerate(all_users)}
        self.item2idx_ = {m: i for i, m in enumerate(all_items)}
        n_users = len(all_users)
        n_items = len(all_items)
        k = self.n_factors

        self.mu_ = train_df[target_col].mean()
        self.bu_ = np.zeros(n_users)
        self.bi_ = np.zeros(n_items)
        self.P_  = self.rng.normal(0, 0.1, (n_users, k))
        self.Q_  = self.rng.normal(0, 0.1, (n_items, k))
        self.Y_  = self.rng.normal(0, 0.1, (n_items, k))  # implicit factor

        # Build implicit feedback sets N(u)
        if implicit_df is not None:
            imp_source = implicit_df
        else:
            imp_source = train_df

        self.Nu_ = {}  # user_idx → list of item_idxs rated
        for uid, group in imp_source.groupby(user_col):
            u_idx = self.user2idx_.get(uid, -1)
            if u_idx < 0:
                continue
            item_idxs = [self.item2idx_[m] for m in group[item_col] if m in self.item2idx_]
            if item_idxs:
                self.Nu_[u_idx] = np.array(item_idxs, dtype=int)

        # Precompute implicit sum for each user
        def _implicit_sum(u_idx):
            if u_idx not in self.Nu_ or len(self.Nu_[u_idx]) == 0:
                return np.zeros(k), 0.0
            nu = self.Nu_[u_idx]
            n_sqrt_inv = 1.0 / np.sqrt(len(nu))
            return self.Y_[nu].sum(axis=0) * n_sqrt_inv, n_sqrt_inv

        u_arr = train_df[user_col].map(self.user2idx_).values
        i_arr = train_df[item_col].map(self.item2idx_).values
        r_arr = train_df[target_col].values.astype(float)

        self.train_rmse_history_ = []
        self.val_rmse_history_   = []
        best_val_rmse = float("inf")
        best_state = None
        no_improve = 0

        for epoch in tqdm(range(self.n_epochs), desc="Training SVD++"):
            idx = self.rng.permutation(len(r_arr))
            u_s, i_s, r_s = u_arr[idx], i_arr[idx], r_arr[idx]

            for u_idx, i_idx, r in zip(u_s, i_s, r_s):
                imp_sum, n_sqrt_inv = _implicit_sum(u_idx)
                pu_hat = self.P_[u_idx] + imp_sum
                pred = self.mu_ + self.bu_[u_idx] + self.bi_[i_idx] + pu_hat @ self.Q_[i_idx]
                err  = r - pred

                self.bu_[u_idx] += self.lr * (err - self.reg * self.bu_[u_idx])
                self.bi_[i_idx] += self.lr * (err - self.reg * self.bi_[i_idx])

                qi_old = self.Q_[i_idx].copy()
                self.P_[u_idx]  += self.lr * (err * qi_old - self.reg * self.P_[u_idx])
                self.Q_[i_idx]  += self.lr * (err * pu_hat  - self.reg * self.Q_[i_idx])

                # Update Y (implicit factors)
                if u_idx in self.Nu_ and len(self.Nu_[u_idx]) > 0:
                    grad_y = err * n_sqrt_inv * qi_old
                    self.Y_[self.Nu_[u_idx]] += self.lr * (
                        grad_y[np.newaxis, :] - self.reg * self.Y_[self.Nu_[u_idx]]
                    )

            train_preds = self._predict_batch(train_df, user_col, item_col)
            self.train_rmse_history_.append(rmse(r_arr, train_preds))

            if val_df is not None and len(val_df) > 0:
                y_val_ep  = val_df[target_col].values
                val_preds = self._predict_batch(val_df, user_col, item_col)
                val_r = rmse(y_val_ep, clip_preds(val_preds))
                self.val_rmse_history_.append(val_r)

                if val_r < best_val_rmse - 1e-5:
                    best_val_rmse = val_r
                    best_state = (self.bu_.copy(), self.bi_.copy(),
                                  self.P_.copy(), self.Q_.copy(), self.Y_.copy())
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= self.patience:
                        print(f"  Early stopping at epoch {epoch+1} (best val RMSE={best_val_rmse:.4f})")
                        break

        if best_state is not None:
            self.bu_, self.bi_, self.P_, self.Q_, self.Y_ = best_state
        return self

    def _predict_batch(self, df, user_col="userId", item_col="movieId"):
        """Vectorized batch prediction for SVD++ (no Python for-loop)."""
        u_idxs = df[user_col].map(self.user2idx_).fillna(-1).astype(int).values
        i_idxs = df[item_col].map(self.item2idx_).fillna(-1).astype(int).values
        preds = np.full(len(df), self.mu_)

        valid = (u_idxs >= 0) & (i_idxs >= 0)
        valid_pos = np.where(valid)[0]
        u_v, i_v = u_idxs[valid], i_idxs[valid]

        if len(u_v) > 0:
            # Vectorized implicit sum: compute per-user imp_sum as weighted Y row sums
            # Stack P + imp_sum for all valid users at once
            imp_sums = np.zeros((len(u_v), self.n_factors))
            for k, u_idx in enumerate(u_v):
                if u_idx in self.Nu_ and len(self.Nu_[u_idx]) > 0:
                    nu = self.Nu_[u_idx]
                    imp_sums[k] = self.Y_[nu].sum(axis=0) / np.sqrt(len(nu))
            pu_hat = self.P_[u_v] + imp_sums
            preds[valid_pos] = (self.mu_ + self.bu_[u_v] + self.bi_[i_v]
                                + np.sum(pu_hat * self.Q_[i_v], axis=1))

        only_u = (u_idxs >= 0) & (i_idxs < 0)
        preds[only_u] = self.mu_ + self.bu_[u_idxs[only_u]]

        only_i = (u_idxs < 0) & (i_idxs >= 0)
        preds[only_i] = self.mu_ + self.bi_[i_idxs[only_i]]

        return np.clip(preds, self.clip_min, self.clip_max)

    def predict(self, df, user_col="userId", item_col="movieId"):
        return self._predict_batch(df, user_col, item_col)

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping Level 3 (BiasedMF).")
else:
    print("Training Level 3: BiasedMF (SGD) ...")
    bmf = BiasedMF(n_factors=20, n_epochs=25, lr=0.005, reg=0.02,
                   patience=5, random_state=RANDOM_SEED)
    bmf.fit(train_df, val_df=val_df, target_col=TARGET)
    print("Training complete.")

    bmf_val_preds  = bmf.predict(val_df)
    bmf_test_preds = bmf.predict(test_df)

    res = evaluate("L3: BiasedMF (val)",  y_val,  bmf_val_preds)
    rq2_results.append({**res, "split": "val",  "level": 3})
    res = evaluate("L3: BiasedMF (test)", y_test, bmf_test_preds)
    rq2_results.append({**res, "split": "test", "level": 3})
    print(f"Level 3 – BiasedMF: Val RMSE={rq2_results[-2]['RMSE']:.4f}  Test RMSE={rq2_results[-1]['RMSE']:.4f}")

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping Level 4 (SVD++).")
else:
    print("Training Level 4: SVD++ (implicit feedback from rating history) ...")
    svdpp = SVDpp(n_factors=20, n_epochs=25, lr=0.005, reg=0.02,
                  patience=5, random_state=RANDOM_SEED)
    svdpp.fit(train_df, val_df=val_df, target_col=TARGET,
              implicit_df=ratings_raw[["userId", "movieId"]])
    print("Training complete.")

    svdpp_val_preds  = svdpp.predict(val_df)
    svdpp_test_preds = svdpp.predict(test_df)

    res = evaluate("L4: SVD++ (val)",  y_val,  svdpp_val_preds)
    rq2_results.append({**res, "split": "val",  "level": 4})
    res = evaluate("L4: SVD++ (test)", y_test, svdpp_test_preds)
    rq2_results.append({**res, "split": "test", "level": 4})
    print(f"Level 4 – SVD++: Val RMSE={rq2_results[-2]['RMSE']:.4f}  Test RMSE={rq2_results[-1]['RMSE']:.4f}")

In [ ]:
if DATA_AVAILABLE:
    # ── RMSE Waterfall / Decomposition Chart ──────────────────────────────
    rq2_df = pd.DataFrame(rq2_results)
    test_rq2 = rq2_df[rq2_df["split"] == "test"].copy()

    # Pick one result per level (prefer additive bias for level 2)
    level_labels = {
        0: "L0: Global Mean",
        1: "L1: User Bias",
        2: "L2b: Additive Bias",
        3: "L3: BiasedMF",
        4: "L4: SVD++",
    }
    level_rows = []
    for lvl, lbl in level_labels.items():
        row = test_rq2[test_rq2["model"].str.startswith(lbl.split(":")[0] + ":")]
        if len(row) == 0:
            row = test_rq2[test_rq2["level"] == lvl]
        if len(row) > 0:
            level_rows.append({"level": lvl, "label": lbl, "RMSE": row.iloc[0]["RMSE"]})

    level_df = pd.DataFrame(level_rows).sort_values("level")

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(level_df["label"], level_df["RMSE"],
                  color=sns.color_palette("muted", len(level_df)))
    ax.set_ylabel("Test RMSE", fontsize=12)
    ax.set_title("RQ2: RMSE Decomposition – Adding Components Step by Step", fontsize=14)
    ax.set_xticklabels(level_df["label"], rotation=20, ha="right")
    for bar, val in zip(bars, level_df["RMSE"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)

    # Annotate improvements
    rmse_vals = level_df["RMSE"].values
    for i in range(1, len(rmse_vals)):
        delta = rmse_vals[i-1] - rmse_vals[i]
        if delta > 0.001:
            ax.annotate(f"−{delta:.4f}", xy=(i, rmse_vals[i] + 0.01),
                        xytext=(i, rmse_vals[i] + 0.04),
                        ha="center", color="green", fontsize=9,
                        arrowprops=dict(arrowstyle="-", color="green"))

    plt.tight_layout()
    plt.show()
    print("\nRMSE Decomposition Table (Test Set):")
    print(level_df[["label", "RMSE"]].to_string(index=False))

In [ ]:
if DATA_AVAILABLE:
    # ── Learning Curves for BiasedMF and SVD++ ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, model, name in [(axes[0], bmf, "BiasedMF"), (axes[1], svdpp, "SVD++")]:
        epochs = range(1, len(model.train_rmse_history_) + 1)
        ax.plot(epochs, model.train_rmse_history_, label="Train RMSE", marker="o", markersize=3)
        if model.val_rmse_history_:
            ax.plot(epochs, model.val_rmse_history_, label="Val RMSE",
                    marker="s", markersize=3, linestyle="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE")
        ax.set_title(f"{name} – Learning Curve", fontsize=13)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping hyperparameter search.")
else:
    print("Running BiasedMF hyperparameter grid search ...")
    print("(Using 15 epochs per config for speed; 27 configs total – may take several minutes)")

    param_grid = {
        "n_factors": [10, 20, 50],
        "lr":        [0.001, 0.005, 0.01],
        "reg":       [0.01, 0.02, 0.05],
    }

    gs_results = []
    for n_f, lr_val, reg_val in product(param_grid["n_factors"],
                                         param_grid["lr"],
                                         param_grid["reg"]):
        m = BiasedMF(n_factors=n_f, n_epochs=15, lr=lr_val, reg=reg_val,
                     patience=5, random_state=RANDOM_SEED)
        m.fit(train_df, val_df=val_df, target_col=TARGET)
        val_r = evaluate("gs", y_val, m.predict(val_df))["RMSE"]
        gs_results.append({"n_factors": n_f, "lr": lr_val, "reg": reg_val, "val_RMSE": val_r})

    gs_df = pd.DataFrame(gs_results)
    best_row = gs_df.loc[gs_df["val_RMSE"].idxmin()]
    print(f"\nBest config: n_factors={best_row['n_factors']:.0f}, "
          f"lr={best_row['lr']}, reg={best_row['reg']} → Val RMSE={best_row['val_RMSE']:.4f}")

    # Heatmap: lr vs reg, faceted by n_factors
    fig, axes = plt.subplots(1, len(param_grid["n_factors"]),
                              figsize=(5 * len(param_grid["n_factors"]), 4), sharey=True)
    for ax, nf in zip(axes, param_grid["n_factors"]):
        pivot = (gs_df[gs_df["n_factors"] == nf]
                 .pivot(index="lr", columns="reg", values="val_RMSE"))
        sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd_r", ax=ax,
                    cbar=(nf == param_grid["n_factors"][-1]))
        ax.set_title(f"n_factors={nf}", fontsize=12)
        ax.set_xlabel("reg"); ax.set_ylabel("lr" if nf == param_grid["n_factors"][0] else "")

    plt.suptitle("RQ2: BiasedMF Hyperparameter Grid Search (Val RMSE)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Section 5 – RQ3: Belief → Actual Rating Prediction

> *Does belief rating add value as a supplementary feature to predict actual ratings? Does combining belief + collaborative filtering outperform pure collaborative filtering?*

### Approach
1. Find matched (user, movie) pairs where the user had a belief AND later actually rated the movie
2. Only include pairs where `belief.tstamp < rating.timestamp` (temporal validity)
3. Compare models with and without belief as a feature:
   - **Naive**: use belief directly as predicted actual rating
   - **Ridge WITHOUT belief** (traditional CF proxy)
   - **Ridge WITH belief**
   - **RF WITHOUT belief**
   - **RF WITH belief**

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping RQ3.")
else:
    # ── Step 1: Find matched pairs ─────────────────────────────────────────
    belief_subset = belief_raw[
        (belief_raw["isSeen"] == 0) &
        (belief_raw["userPredictRating"].notna())
    ][["userId", "movieId", "userPredictRating", "tstamp"]].copy()
    belief_subset = belief_subset.rename(columns={"userPredictRating": "belief_rating",
                                                   "tstamp": "belief_tstamp"})

    ratings_subset = ratings_raw[["userId", "movieId", "rating", "timestamp"]].copy()
    ratings_subset = ratings_subset.rename(columns={"rating": "actual_rating",
                                                      "timestamp": "actual_timestamp"})

    matched = belief_subset.merge(ratings_subset, on=["userId", "movieId"], how="inner")

    # ── Step 2: Temporal filter – belief must precede actual rating ────────
    if "belief_tstamp" in matched.columns and "actual_timestamp" in matched.columns:
        # Ensure same dtype for comparison (try numeric comparison)
        try:
            matched = matched[matched["belief_tstamp"] < matched["actual_timestamp"]]
            print(f"After temporal filter (belief_tstamp < actual_timestamp): {len(matched):,} rows")
        except TypeError:
            print("[INFO] Could not compare timestamps (dtype mismatch) – skipping temporal filter.\n"
                  "  Tip: convert timestamp columns to numeric with pd.to_numeric(df[col], errors='coerce')")

    n_matched = len(matched)
    print(f"Matched (user, movie) pairs with belief AND later actual rating: {n_matched:,}")

    if n_matched < 100:
        print(f"\n⚠ WARNING: Only {n_matched} matched pairs found.")
        print("  This is expected if the belief elicitation and rating collection windows don't overlap much.")
        print("  Results below should be interpreted with caution due to small sample size.")

In [ ]:
if DATA_AVAILABLE and n_matched > 0:
    matched["belief_gap_actual"] = matched["actual_rating"] - matched["belief_rating"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.hist(matched["belief_gap_actual"], bins=20,
            color=sns.color_palette("muted")[0], edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Actual Rating − Belief Rating")
    ax.set_ylabel("Count")
    ax.set_title("RQ3: Distribution of Belief Gap (actual − belief)", fontsize=13)

    ax = axes[1]
    ax.scatter(matched["belief_rating"], matched["actual_rating"],
               alpha=0.3, s=15, color=sns.color_palette("muted")[1])
    lo, hi = 0.5, 5.5
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Perfect belief=actual")
    ax.set_xlabel("Belief Rating"); ax.set_ylabel("Actual Rating")
    ax.set_title("RQ3: Belief vs Actual Rating", fontsize=13)
    ax.legend()
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

    plt.tight_layout()
    plt.show()

    print(f"\nMean belief gap (actual - belief): {matched['belief_gap_actual'].mean():+.3f}")
    print(f"Correlation (belief, actual): {matched[['belief_rating','actual_rating']].corr().iloc[0,1]:.3f}")

In [ ]:
if DATA_AVAILABLE and n_matched > 0:
    # ── Step 4: Prepare features ───────────────────────────────────────────
    rq3_df = matched.copy()

    # Add genre one-hot
    rq3_df = rq3_df.merge(
        movies_raw[["movieId", "genres"]],
        on="movieId", how="left"
    )
    for g in GENRE_COLS:
        rq3_df[f"genre_{g}"] = rq3_df["genres"].str.contains(g, regex=False, na=False).astype(int)

    # Add user/movie stats
    rq3_df = rq3_df.merge(user_stats,  on="userId",  how="left")
    rq3_df = rq3_df.merge(movie_stats, on="movieId", how="left")

    FEAT_BASE = (
        [f"genre_{g}" for g in GENRE_COLS]
        + ["user_mean_rating", "user_rating_count", "user_rating_std",
           "movie_mean_rating", "movie_rating_count"]
    )
    FEAT_BASE = [c for c in FEAT_BASE if c in rq3_df.columns]
    FEAT_BELIEF = FEAT_BASE + ["belief_rating"]

    # ── Step 5: Time-based split on matched pairs ──────────────────────────
    sort_col = "belief_tstamp" if "belief_tstamp" in rq3_df.columns else rq3_df.columns[0]
    rq3_df_sorted = rq3_df.sort_values(sort_col).reset_index(drop=True)

    n_rq3 = len(rq3_df_sorted)
    n_tr = max(1, int(n_rq3 * 0.70))
    n_va = max(1, int(n_rq3 * 0.15))

    rq3_train = rq3_df_sorted.iloc[:n_tr]
    rq3_val   = rq3_df_sorted.iloc[n_tr:n_tr + n_va]
    rq3_test  = rq3_df_sorted.iloc[n_tr + n_va:]

    if len(rq3_test) == 0:
        rq3_test = rq3_val

    y_rq3_train = rq3_train["actual_rating"].values
    y_rq3_val   = rq3_val["actual_rating"].values
    y_rq3_test  = rq3_test["actual_rating"].values

    print(f"RQ3 split — Train: {len(rq3_train)}  Val: {len(rq3_val)}  Test: {len(rq3_test)}")

In [ ]:
if DATA_AVAILABLE and n_matched > 0 and len(rq3_test) > 0:
    rq3_results = []

    # ── Naive: belief directly as prediction ──────────────────────────────
    naive_preds = clip_preds(rq3_test["belief_rating"].values)
    res = evaluate("Naive (belief=actual)", y_rq3_test, naive_preds)
    rq3_results.append({**res, "has_belief": False, "model_type": "Naive"})

    # ── Ridge WITHOUT belief ───────────────────────────────────────────────
    X_tr_nb = rq3_train[FEAT_BASE].fillna(0).values
    X_te_nb = rq3_test[FEAT_BASE].fillna(0).values

    ridge_nb = Ridge(alpha=1.0, random_state=RANDOM_SEED)
    ridge_nb.fit(X_tr_nb, y_rq3_train)
    res = evaluate("Ridge (no belief)", y_rq3_test, ridge_nb.predict(X_te_nb))
    rq3_results.append({**res, "has_belief": False, "model_type": "Ridge"})

    # ── Ridge WITH belief ──────────────────────────────────────────────────
    X_tr_wb = rq3_train[FEAT_BELIEF].fillna(0).values
    X_te_wb = rq3_test[FEAT_BELIEF].fillna(0).values

    ridge_wb = Ridge(alpha=1.0, random_state=RANDOM_SEED)
    ridge_wb.fit(X_tr_wb, y_rq3_train)
    res = evaluate("Ridge + belief", y_rq3_test, ridge_wb.predict(X_te_wb))
    rq3_results.append({**res, "has_belief": True, "model_type": "Ridge"})

    # ── RF WITHOUT belief ──────────────────────────────────────────────────
    rf_nb = RandomForestRegressor(n_estimators=100, max_depth=8, n_jobs=-1, random_state=RANDOM_SEED)
    rf_nb.fit(X_tr_nb, y_rq3_train)
    res = evaluate("RF (no belief)", y_rq3_test, rf_nb.predict(X_te_nb))
    rq3_results.append({**res, "has_belief": False, "model_type": "RF"})

    # ── RF WITH belief ─────────────────────────────────────────────────────
    rf_wb = RandomForestRegressor(n_estimators=100, max_depth=8, n_jobs=-1, random_state=RANDOM_SEED)
    rf_wb.fit(X_tr_wb, y_rq3_train)
    res = evaluate("RF + belief", y_rq3_test, rf_wb.predict(X_te_wb))
    rq3_results.append({**res, "has_belief": True, "model_type": "RF"})

    rq3_df_res = pd.DataFrame(rq3_results)
    print("\nRQ3 Results (Test Set):")
    display(rq3_df_res[["model", "RMSE", "MAE"]])

In [ ]:
if DATA_AVAILABLE and n_matched > 0 and len(rq3_test) > 0:
    # ── RMSE comparison bar chart ──────────────────────────────────────────
    rq3_sorted = rq3_df_res.sort_values("RMSE")
    colors_rq3 = ["#d73027" if not hb else "#1a9850"
                  for hb in rq3_sorted["has_belief"]]

    plt.figure(figsize=(9, 5))
    bars = plt.barh(rq3_sorted["model"], rq3_sorted["RMSE"], color=colors_rq3)
    plt.xlabel("Test RMSE", fontsize=12)
    plt.title("RQ3: RMSE Comparison – With vs Without Belief Feature\n(Red=no belief, Green=with belief)", fontsize=13)
    plt.axvline(rq3_sorted["RMSE"].min(), color="gray", linestyle=":", linewidth=1)
    for bar, val in zip(bars, rq3_sorted["RMSE"]):
        plt.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{val:.4f}", va="center", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
if DATA_AVAILABLE and n_matched > 0 and len(rq3_test) > 0:
    # ── RF Feature Importance ─────────────────────────────────────────────
    fi = pd.Series(rf_wb.feature_importances_, index=FEAT_BELIEF).sort_values(ascending=True)
    top_fi = fi.tail(20)

    colors_fi = ["#e41a1c" if "belief" in idx else sns.color_palette("muted")[0]
                 for idx in top_fi.index]

    plt.figure(figsize=(10, 6))
    plt.barh(top_fi.index, top_fi.values, color=colors_fi)
    plt.xlabel("Feature Importance (impurity)", fontsize=12)
    plt.title("RQ3: RF Feature Importance (red = belief_rating)", fontsize=13)
    plt.tight_layout()
    plt.show()

    belief_rank = (fi.rank(ascending=False)["belief_rating"]
                   if "belief_rating" in fi.index else "N/A")
    print(f"\nbelief_rating importance rank: {belief_rank:.0f} / {len(fi)}")

    # ── Ridge Coefficients ────────────────────────────────────────────────
    coef = pd.Series(ridge_wb.coef_, index=FEAT_BELIEF).sort_values()
    top_coef = pd.concat([coef.head(10), coef.tail(10)]).drop_duplicates()

    plt.figure(figsize=(10, 5))
    colors_coef = ["#d73027" if v < 0 else "#1a9850" for v in top_coef.values]
    plt.barh(top_coef.index, top_coef.values, color=colors_coef)
    plt.axvline(0, color="black", linewidth=1.0, linestyle="--")
    plt.xlabel("Coefficient Value", fontsize=12)
    plt.title("RQ3: Ridge Regression Coefficients (top features)", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Section 6 – Ablation: Leakage Demo

### Why `systemPredictRating` is excluded

`systemPredictRating` in `belief_data.csv` is the output of the dataset's internal recommender system shown to users **before** they provide their belief rating. This creates **policy leakage**: the variable is computed by the same system whose behavior we are studying.

Including it gives an artificially inflated RMSE improvement — it's not a generalizable feature.

This section demonstrates this leakage effect for educational purposes.

In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping ablation section.")
else:
    if "systemPredictRating" in train_df.columns:
        feat_no_leak = [c for c in FEAT_BASE if c in train_df.columns] if 'FEAT_BASE' in dir() else []
        if not feat_no_leak:
            print("[INFO] FEAT_BASE not defined – run preprocessing cells first.")
        else:
            # Ridge WITHOUT system score
            X_tr_nl = train_df[feat_no_leak].fillna(0).values
            X_ts_nl = test_df[feat_no_leak].fillna(0).values
            ridge_nl = Ridge(alpha=1.0, random_state=RANDOM_SEED)
            ridge_nl.fit(X_tr_nl, y_train)
            res_no_leak = evaluate("Ridge (no systemPredictRating)", y_test, ridge_nl.predict(X_ts_nl))

            feat_leak = feat_no_leak + ["systemPredictRating"]
            X_tr_leak = train_df[feat_leak].fillna(0).values
            X_ts_leak = test_df[feat_leak].fillna(0).values
            ridge_leak = Ridge(alpha=1.0, random_state=RANDOM_SEED)
            ridge_leak.fit(X_tr_leak, y_train)
            res_leak = evaluate("Ridge + systemPredictRating (LEAKED)", y_test, ridge_leak.predict(X_ts_leak))

            ablation_df = pd.DataFrame([res_no_leak, res_leak])
            print("=== Ablation: Leakage Demo ===")
            display(ablation_df[["model", "RMSE", "MAE"]])
            improvement = res_no_leak["RMSE"] - res_leak["RMSE"]
            print(f"\nRMSE improvement from leaky feature: {improvement:.4f}")
            print("→ This improvement is NOT trustworthy – systemPredictRating is computed from the same recommender.")
    else:
        print("[INFO] systemPredictRating column not found – leakage demo skipped.")
        print("The column may have been removed from the dataset.")

---
## Section 7 – Conclusion

### Summary of Findings

This notebook answered 3 research questions about belief ratings in the MovieLens Beliefs dataset.

In [ ]:
print("=" * 70)
print("CONCLUSION – Belief Rating Prediction")
print("=" * 70)

if DATA_AVAILABLE:
    print("\n── RQ1: Belief Gap by Genre ────────────────────────────────────────")
    print("  Belief gap = userPredictRating − actual community mean rating")
    if 'genre_gap_df' in dir() and len(genre_gap_df) > 0:
        top_over  = genre_gap_df.tail(3)[::-1]
        top_under = genre_gap_df.head(3)
        print(f"  Most overestimated:  {', '.join(top_over.index.tolist())}")
        print(f"  Most underestimated: {', '.join(top_under.index.tolist())}")
        print(f"  Overall mean gap: {rq1_df_valid['belief_gap'].mean():+.3f}")
    else:
        print("  (RQ1 data not computed)")

    print("\n── RQ2: Belief Formation Decomposition ─────────────────────────────")
    if 'level_df' in dir() and len(level_df) > 0:
        rmse_l0 = level_df[level_df["level"] == 0]["RMSE"].values
        rmse_l3 = level_df[level_df["level"] == 3]["RMSE"].values
        if len(rmse_l0) > 0 and len(rmse_l3) > 0:
            total_drop = rmse_l0[0] - rmse_l3[0]
            print(f"  RMSE drop from Global Mean → BiasedMF: {total_drop:.4f}")
        print("  Key insight: each component (user bias, item bias, latent factors)")
        print("  contributes incrementally to explaining belief formation.")
    else:
        print("  (RQ2 data not computed)")

    print("\n── RQ3: Belief as Predictor of Actual Rating ───────────────────────")
    if 'rq3_df_res' in dir() and len(rq3_df_res) > 0:
        r_nb = rq3_df_res[rq3_df_res["model"].str.contains("no belief")]["RMSE"]
        r_wb = rq3_df_res[rq3_df_res["model"].str.contains("+ belief", regex=False)]["RMSE"]
        if len(r_nb) > 0 and len(r_wb) > 0:
            improvement = r_nb.min() - r_wb.min()
            print(f"  Best RMSE without belief: {r_nb.min():.4f}")
            print(f"  Best RMSE with belief:    {r_wb.min():.4f}")
            print(f"  Improvement from belief feature: {improvement:+.4f}")
            verdict = "YES" if improvement > 0.01 else ("MARGINAL" if improvement > 0 else "NO")
            print(f"  → Does belief help? {verdict}")
    else:
        print("  (RQ3 matched pairs not found or insufficient data)")

    print("\n── Limitations ──────────────────────────────────────────────────────")
    print("  - Selection bias: only elicitation study participants")
    print("  - MNAR: isSeen=-1 rows are excluded")
    print("  - RQ3 matched pairs may be limited (depends on dataset overlap)")
    print("  - systemPredictRating excluded (policy leakage)")
    print("  - No external metadata (e.g., cast, director, plot)")

    print("\n── Future Work ──────────────────────────────────────────────────────")
    print("  - Neural Collaborative Filtering (NCF) for RQ2 Level 5")
    print("  - Temporal dynamics: how do beliefs change over time?")
    print("  - Cross-genre belief calibration personalization")
    print("  - Integration of external knowledge (TMDB, IMDb)")
else:
    print("Data not available – cannot generate conclusions.")